Step 1 — Set Up Google Colab

In [ ]:
!pip install -U transformers datasets peft accelerate
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


Step 2 — Import Libraries

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

import torch
import pandas as pd
import time

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


Step 2 – Load the Qwen Model and Tokenizer

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

print(f"Base model parameters: {model.num_parameters():,}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model parameters: 494,032,768


Step 3 – Prepare a Small Task-Specific Dataset

In [ ]:
from datasets import load_dataset, DatasetDict

# Load Recipe Dataset
print("Loading Recipe Dataset...")
ds_recipes = load_dataset("SandhyaKilari/RecipeNLG_dataset")

# Select only 500 samples from the training set
selected_recipes = ds_recipes['train'].select(range(500))

# Let's write a function to format the recipe dataset into structured text
def format_recipe(example):
    title = example.get('title', 'Unknown Recipe')
    ingredients = example.get('ingredients', 'Unknown Ingredients')
    directions = example.get('directions', 'Unknown Directions')

    formatted_text = f"Recipe: {title}\nIngredients: {ingredients}\nDirections: {directions}"
    return {"text": formatted_text}

print("Formatting recipes...")
train_recipes = selected_recipes.map(format_recipe, remove_columns=selected_recipes.column_names)

# Create a DatasetDict with the formatted recipes
ds = DatasetDict({
    'train': train_recipes.shuffle(seed=42)
})

print("Dataset loaded, formatted, and shuffled with 500 samples.")
print(ds)
print(ds['train'][0]['text'])

Loading Recipe Dataset...
Formatting recipes...
Dataset loaded, formatted, and shuffled with 500 samples.
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 500
    })
})
Recipe: Yum Yum Salad
Ingredients: ["1 can cherry pie filling", "1 large can drained, crushed pineapple", "1 can Borden milk", "3/4 c. chopped pecans", "1 (8 oz.) Cool Whip"]
Directions: ["Mix together and let stand overnight in refrigerator."]


Step 4 – Format and Tokenize the Data

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )


tokenized = ds["train"].map(
    tokenize_function,
    batched=True
)

print(tokenized)

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 500
})


In [ ]:
print(tokenized[0])

{'text': 'Recipe: Yum Yum Salad\nIngredients: ["1 can cherry pie filling", "1 large can drained, crushed pineapple", "1 can Borden milk", "3/4 c. chopped pecans", "1 (8 oz.) Cool Whip"]\nDirections: ["Mix together and let stand overnight in refrigerator."]', 'input_ids': [28780, 25, 809, 372, 809, 372, 63546, 198, 45747, 25, 4383, 16, 646, 40880, 4349, 21274, 497, 330, 16, 3460, 646, 60408, 11, 32645, 77082, 497, 330, 16, 646, 425, 27784, 14074, 497, 330, 18, 14, 19, 272, 13, 37425, 30000, 596, 497, 330, 16, 320, 23, 24596, 6138, 23931, 94302, 7026, 60324, 25, 4383, 58083, 3786, 323, 1077, 2498, 24398, 304, 44944, 92104], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator created.")

Data collator created.


Step 5 – Configure LoRA

In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

Step 6 – Attach the LoRA Adapter

In [ ]:
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


Step 7 – Define a Baseline Training Configuration


In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen_lora_output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

Step 8 – Run Multiple Hyperparameter Configurations

In [ ]:
def run_experiment(run_name, learning_rate, epochs, lora_r):

    print("=" * 70)
    print(f"STARTING {run_name}")
    print("=" * 70)

    print("Learning Rate:", learning_rate)
    print("Epochs:", epochs)
    print("LoRA Rank:", lora_r)

    # Load a fresh base model
    model = AutoModelForCausalLM.from_pretrained(model_id)

    # Configure LoRA
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

    # Attach LoRA
    model = get_peft_model(model, lora_config)

    model.print_trainable_parameters()

    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"./{run_name}",
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        learning_rate=learning_rate,
        logging_steps=50,
        save_strategy="epoch",
        report_to="none"
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized,
        data_collator=data_collator
    )

    # Start timer
    start_time = time.time()

    train_result = trainer.train()

    end_time = time.time()

    training_time = end_time - start_time

    # Final training loss
    loss = train_result.training_loss

    print("\nTraining completed.")
    print("Final training loss:", loss)
    print("Training time:", training_time, "seconds")

    # Save LoRA adapter
    trainer.save_model(f"./{run_name}")

    result = {
        "Run": run_name,
        "Learning Rate": learning_rate,
        "Epochs": epochs,
        "LoRA r": lora_r,
        "Loss": loss,
        "Time (seconds)": training_time
    }

    # Free memory
    del trainer
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

Run A – Baseline


In [ ]:
result_A = run_experiment(
    run_name="run_A",
    learning_rate=1e-4,
    epochs=2,
    lora_r=4
)

print(result_A)

STARTING run_A
Learning Rate: 0.0001
Epochs: 2
LoRA Rank: 4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 270,336 || all params: 494,303,104 || trainable%: 0.0547


Step,Training Loss
50,2.171296
100,1.910056
150,1.788075
200,1.715064
250,1.738069
300,1.729339
350,1.712700
400,1.698767
450,1.731645
500,1.707252



Training completed.
Final training loss: 1.7902262115478516
Training time: 223.45756077766418 seconds
{'Run': 'run_A', 'Learning Rate': 0.0001, 'Epochs': 2, 'LoRA r': 4, 'Loss': 1.7902262115478516, 'Time (seconds)': 223.45756077766418}


Run A produced a training loss of 1.7902 after two epochs. This serves as the baseline against which the other configurations can be compared.

Run B – Learning Rate


In [ ]:
result_B = run_experiment(
    run_name="run_B",
    learning_rate=2e-4,
    epochs=2,
    lora_r=4
)

print(result_B)

STARTING run_B
Learning Rate: 0.0002
Epochs: 2
LoRA Rank: 4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 270,336 || all params: 494,303,104 || trainable%: 0.0547


Step,Training Loss
50,2.083407
100,1.831360
150,1.740632
200,1.684525
250,1.707838
300,1.686078
350,1.671521
400,1.661102
450,1.690163
500,1.669000



Training completed.
Final training loss: 1.7425626373291017
Training time: 200.7430546283722 seconds
{'Run': 'run_B', 'Learning Rate': 0.0002, 'Epochs': 2, 'LoRA r': 4, 'Loss': 1.7425626373291017, 'Time (seconds)': 200.7430546283722}


Increasing the learning rate from 1e-4 to 2e-4 reduced the training loss from 1.7902 to 1.7426. Training time also decreased from approximately 223.46 seconds to 200.74 seconds. In this experiment, the higher learning rate improved convergence compared with the baseline without producing an observed training failure.

Run C – LoRA Capacity


In [ ]:
result_C = run_experiment(
    run_name="run_C",
    learning_rate=2e-4,
    epochs=2,
    lora_r=8
)

print(result_C)

STARTING run_C
Learning Rate: 0.0002
Epochs: 2
LoRA Rank: 8


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


Step,Training Loss
50,2.085383
100,1.825173
150,1.733365
200,1.678477
250,1.701927
300,1.677010
350,1.663220
400,1.652493
450,1.683242
500,1.662150



Training completed.
Final training loss: 1.7362438507080078
Training time: 199.4520137310028 seconds
{'Run': 'run_C', 'Learning Rate': 0.0002, 'Epochs': 2, 'LoRA r': 8, 'Loss': 1.7362438507080078, 'Time (seconds)': 199.4520137310028}


Increasing the LoRA rank from 4 to 8 slightly reduced the training loss from 1.7426 to 1.7362. The number of trainable parameters doubled from 270,336 to 540,672, providing greater adapter capacity. The training time remained approximately similar.

Run D - High LR

In [ ]:
result_D = run_experiment(
    run_name="run_D_high_lr",
    learning_rate=5e-4,
    epochs=2,
    lora_r=4
)

print(result_D)

STARTING run_D_high_lr
Learning Rate: 0.0005
Epochs: 2
LoRA Rank: 4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 270,336 || all params: 494,303,104 || trainable%: 0.0547


Step,Training Loss
50,1.984231
100,1.782506
150,1.706238
200,1.658887
250,1.678159
300,1.623737
350,1.608652
400,1.599189
450,1.631195
500,1.607744



Training completed.
Final training loss: 1.6880536651611329
Training time: 198.6374430656433 seconds
{'Run': 'run_D_high_lr', 'Learning Rate': 0.0005, 'Epochs': 2, 'LoRA r': 4, 'Loss': 1.6880536651611329, 'Time (seconds)': 198.6374430656433}


The high learning-rate configuration produced the lowest training loss among the six runs, at 1.6881, with a training time of approximately 198.64 seconds. In the recorded training result, the high learning rate did not cause the loss to fail to decrease.

Run E - High Epochs

In [ ]:
result_E = run_experiment(
    run_name="run_E_high_epochs",
    learning_rate=2e-4,
    epochs=3,
    lora_r=4
)

print(result_E)

STARTING run_E_high_epochs
Learning Rate: 0.0002
Epochs: 3
LoRA Rank: 4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 270,336 || all params: 494,303,104 || trainable%: 0.0547


Step,Training Loss
50,2.082614
100,1.829056
150,1.737779
200,1.682376
250,1.705089
300,1.679120
350,1.665089
400,1.655390
450,1.685372
500,1.661084



Training completed.
Final training loss: 1.7004680582682292
Training time: 296.28163623809814 seconds
{'Run': 'run_E_high_epochs', 'Learning Rate': 0.0002, 'Epochs': 3, 'LoRA r': 4, 'Loss': 1.7004680582682292, 'Time (seconds)': 296.28163623809814}


Increasing the number of epochs from 2 to 3 reduced the training loss from 1.7426 to 1.7005. However, training time increased substantially from approximately 200.74 seconds to 296.28 seconds. The lower loss indicates additional adaptation to the training data.

Run F - LOW LoRA RANK

In [ ]:
result_F = run_experiment(
    run_name="run_F_low_rank",
    learning_rate=2e-4,
    epochs=2,
    lora_r=2
)

print(result_F)

STARTING run_F_low_rank
Learning Rate: 0.0002
Epochs: 2
LoRA Rank: 2


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 135,168 || all params: 494,167,936 || trainable%: 0.0274


Step,Training Loss
50,2.087708
100,1.832340
150,1.738736
200,1.685003
250,1.709157
300,1.687143
350,1.673931
400,1.663636
450,1.697801
500,1.674233



Training completed.
Final training loss: 1.7449687347412108
Training time: 204.27929663658142 seconds
{'Run': 'run_F_low_rank', 'Learning Rate': 0.0002, 'Epochs': 2, 'LoRA r': 2, 'Loss': 1.7449687347412108, 'Time (seconds)': 204.27929663658142}


Reducing the LoRA rank from 4 to 2 reduced the number of trainable parameters from 270,336 to 135,168. The training loss increased slightly from 1.7426 to 1.7442, suggesting that the lower-capacity adapter captured slightly less task-specific information in this experiment. The training time was also slightly lower.

Step 9 – Evaluate Task-Specific Performance



In [ ]:
evaluation_prompts = [
    "Recipe: Vegetable Pasta\nDirections:",
    "Recipe: Chocolate Pancakes\nDirections:",
    "Recipe: Tomato Soup\nDirections:"
]

for i, prompt in enumerate(evaluation_prompts, 1):
    print(f"{i}. {prompt}")

1. Recipe: Vegetable Pasta
Directions:
2. Recipe: Chocolate Pancakes
Directions:
3. Recipe: Tomato Soup
Directions:


In [ ]:
def generate_response(model, prompt):

    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [ ]:
from peft import PeftModel

Evaluate Run A

In [ ]:
model_A = AutoModelForCausalLM.from_pretrained(model_id)

model_A = PeftModel.from_pretrained(
    model_A,
    "./run_A"
)

model_A = model_A.to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

for prompt in evaluation_prompts:

    print("=" * 60)
    print("PROMPT:")
    print(prompt)

    print("\nRUN A:")
    print(generate_response(model_A, prompt))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT:
Recipe: Vegetable Pasta
Directions:

RUN A:
Recipe: Vegetable Pasta
Directions: Bring 1/2 cup of water to a boil in large pot. Add pasta and cook until al dente, about 5 minutes. Drain, add oil (or vegetable oil) and cook for 30 seconds or so on medium heat. Add cheese and cook another minute.
Source: "The Complete Idiot's Guide to Italian Cooking" by Nancy Lippman. Copyright © 2004. Used by permission of Simon & Schuster, Inc., New York. All rights reserved.
PROMPT:
Recipe: Chocolate Pancakes
Directions:

RUN A:
Recipe: Chocolate Pancakes
Directions: Beat eggs, sugar and vanilla. Add flour to make batter. Pour into a greased pan and cook at 350\u00b0 for 15 minutes. Sprinkle with chocolate chips.
How to Prepare: Put butter in saucepan and boil until melted. Mix with cream of tartar and vanilla extract. Cool slightly before adding milk and sugar. Stir in chocolate chips. Pour into pan. Bake at 350\u00b0 for 25-30 minutes or until top is browned and syrup has set. Serve warm wit

The responses are mostly relevant and follow the recipe format.
The recipes contain useful cooking directions, but some outputs include unnecessary source information and extra recipe details.
Overall, the responses are understandable but not completely consistent.

Evaluate Run B

In [ ]:
model_B = AutoModelForCausalLM.from_pretrained(model_id)

model_B = PeftModel.from_pretrained(
    model_B,
    "./run_B"
)

model_B = model_B.to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

for prompt in evaluation_prompts:

    print("=" * 60)
    print("PROMPT:")
    print(prompt)

    print("\nRUN B:")
    print(generate_response(model_B, prompt))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT:
Recipe: Vegetable Pasta
Directions:

RUN B:
Recipe: Vegetable Pasta
Directions: 1. Drain and rinse pasta, then add water to cover it. Bring to a boil in boiling water until just tender when you pull off the noodles. Drain and set aside; reserve 1/2 cup of pasta cooking liquid.
2. Cook vegetables as directed on package instructions (if using). Drain and set aside.
3. In large saucepan, combine water and vegetable cooking liquid. Bring to a boil over high heat, stirring occasionally, for 5 minutes. Remove from heat and let cool slightly before draining.
4. Add cooked pasta to hot soup and simmer until heated through. Stir in reserved pasta cooking liquid and stir well to mix all ingredients. Serve immediately. Makes 6 servings. Based on the passage above, what are
PROMPT:
Recipe: Chocolate Pancakes
Directions:

RUN B:
Recipe: Chocolate Pancakes
Directions: Mix together butter and sugar. Add eggs, vanilla, and baking soda. Pour into a greased 9 x 13-inch pan; bake at 350\u00b0 for

The responses are generally clear and structured.
However, some outputs contain unusual information, such as an unrealistic pancake quantity.
The model also starts generating unrelated content after the recipe, showing some instruction-following issues.

Evaluate Run C

In [ ]:
model_C = AutoModelForCausalLM.from_pretrained(model_id)

model_C = PeftModel.from_pretrained(
    model_C,
    "./run_C"
)

model_C = model_C.to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

for prompt in evaluation_prompts:

    print("=" * 60)
    print("PROMPT:")
    print(prompt)

    print("\nRUN C:")
    print(generate_response(model_C, prompt))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT:
Recipe: Vegetable Pasta
Directions:

RUN C:
Recipe: Vegetable Pasta
Directions: Cook pasta according to package directions. Drain and mix in 2 cans of tomato sauce, 1 can cream cheese (or 4 ounces whole milk), 3 Tbsp. olive oil, 1/2 tsp. salt, and pepper. Stir well. Spread evenly on a lightly greased baking sheet. Bake at 350\u00b0 for about 20 minutes or until golden brown.
Nutritional Information (per serving): Calories: 68, Total Fat: 12g Saturated Fat: 2. 9g Cholesterol: 170mg Sodium: 510 mg Protein: 16 g
Ingredients: [canned tomato sauce, cream cheese, olive oil
PROMPT:
Recipe: Chocolate Pancakes
Directions:

RUN C:
Recipe: Chocolate Pancakes
Directions: In a large mixing bowl, combine sugar, eggs, vanilla and flour. Add milk, oil and salt. Mix well. Pour into 12 muffin cups. Bake at 350\u00b0 for 25 minutes or until golden brown. Cool in pans on racks before serving.
Nutritional Information (per serving): Calories: 460 Total Fat: 8g Saturated Fat: 7g Cholesterol: 90mg Sod

The responses are coherent and mostly follow the expected recipe format.
Some outputs include unnecessary nutritional information and inconsistent serving details.
Overall, the quality is fairly consistent, with only a small improvement compared with Run B.

Evaluate Run D

In [ ]:
model_D = AutoModelForCausalLM.from_pretrained(model_id)

model_D = PeftModel.from_pretrained(
    model_D,
    "./run_D_high_lr"
)

model_D = model_D.to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

for prompt in evaluation_prompts:

    print("=" * 60)
    print("PROMPT:")
    print(prompt)

    print("\nRUN D - HIGH LR:")
    print(generate_response(model_D, prompt))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT:
Recipe: Vegetable Pasta
Directions:

RUN D - HIGH LR:
Recipe: Vegetable Pasta
Directions: Combine flour, water and oil in a saucepan. Cook until thickened. Stir in salt, pepper and garlic; cook 1 to 2 minutes. Add eggs and stir well. Mix with pasta, then sprinkle with cheese (optional). Bake at 400\u00b0 for 30 to 5 minutes or until cheese is melted and bubbly.
Makes 8 servings. Adapted from "The Complete Italian Cookbook" by Mary Beth Dye, published by William Morrow & Company, Inc., 1976.
PROMPT:
Recipe: Chocolate Pancakes
Directions:

RUN D - HIGH LR:
Recipe: Chocolate Pancakes
Directions: In a 9 x 13-inch pan, mix brown sugar and flour. Set aside; then in another bowl, beat butter until smooth. Add eggs and vanilla, beating well. Stir in chocolate mixture. Pour into prepared pan; bake at 350\u00b0 for 25 to 30 minutes or until done. Cool completely on wire rack before cutting into squares.
Makes about 6 dozen pancakes. Based on the passage above, what is the temperature ran

The responses show more noticeable errors and inconsistency.
For example, the Vegetable Pasta recipe has an incorrect cooking time, and the Chocolate Pancakes response suddenly answers an unrelated temperature question.
The Tomato Soup response also repeats instructions.
Overall, the high learning rate causes more unwanted or unstable content.

Evaluate Run E

In [ ]:
model_E = AutoModelForCausalLM.from_pretrained(model_id)

model_E = PeftModel.from_pretrained(
    model_E,
    "./run_E_high_epochs"
)

model_E = model_E.to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

for prompt in evaluation_prompts:

    print("=" * 60)
    print("PROMPT:")
    print(prompt)

    print("\nRUN E - HIGH EPOCHS:")
    print(generate_response(model_E, prompt))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT:
Recipe: Vegetable Pasta
Directions:

RUN E - HIGH EPOCHS:
Recipe: Vegetable Pasta
Directions: Cook pasta according to package directions. Drain and set aside. Heat oil in a large saucepan over medium heat. Add onion, garlic and mushrooms. Cook until onion is translucent and mushrooms are tender. Stir in tomatoes and basil. Bring to a boil and reduce heat to low and simmer for 10 minutes or until vegetables are tender. Serve with cooked pasta on top.
Nutritional Facts: Per serving (28%), Calories: 543, Total Carbohydrates: 12 g, Total Fat: 7 g, Saturated Fat: 6 g, Cholesterol: 30 mg, Sodium: 136 mg, Total Sugar: 29 g, Fiber: 7 g, Protein: 
PROMPT:
Recipe: Chocolate Pancakes
Directions:

RUN E - HIGH EPOCHS:
Recipe: Chocolate Pancakes
Directions: Combine 1/2 cup sugar, 1/2 c. butter and 3 eggs in a mixing bowl. Add flour and mix well. Pour into prepared pan. Bake at 350\u00b0 for about 25 minutes or until cooked through. Cool completely before slicing. Makes about 8 servings.
PRO

The responses are generally clear and relevant, especially the Chocolate Pancakes response.
However, the model still adds nutritional information and unrelated follow-up content.
The Tomato Soup response goes into an unrelated question about toppings.
Overall, the responses remain coherent but show some unnecessary continuation.

Evaluate Run F

In [ ]:
model_F = AutoModelForCausalLM.from_pretrained(model_id)

model_F = PeftModel.from_pretrained(
    model_F,
    "./run_F_low_rank"
)

model_F = model_F.to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

for prompt in evaluation_prompts:

    print("=" * 60)
    print("PROMPT:")
    print(prompt)

    print("\nRUN F - LOW RANK:")
    print(generate_response(model_F, prompt))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT:
Recipe: Vegetable Pasta
Directions:

RUN F - LOW RANK:
Recipe: Vegetable Pasta
Directions: Cook pasta according to package directions. Drain and set aside. Heat oil in large skillet over medium heat. Add onion, garlic and 2 Tbsp. salt. Cook until onion is soft but not browned (about 3 minutes). Stir in remaining vegetables and cook for another 1 minute or so. Add broth and bring to a boil. Reduce heat to low and simmer, stirring occasionally, about 15 minutes.
Serves 6. Makes 4 servings. Can be made ahead and stored at room temperature for up to two days. Refrigerate if desired. Makes a flavorful vegetable stew that can also be served as a side dish.
PROMPT:
Recipe: Chocolate Pancakes
Directions:

RUN F - LOW RANK:
Recipe: Chocolate Pancakes
Directions: Preheat oven to 350\u00b0. Beat butter, sugar and eggs in a medium bowl until smooth. Add vanilla, baking soda, flour, cocoa and milk. Mix well and pour into prepared pan. Bake at 350\u00b0 for 25 minutes or until set.
Notes: Th

The responses are generally clear and relevant, especially the Chocolate Pancakes response.
However, some outputs have inconsistent serving/quantity information and include unnecessary extra details.
The Tomato Soup response also continues into unrelated content at the end.
Overall, the responses are coherent but show some inconsistency and unnecessary continuation.

STEP 10 — FINAL COMPARISON

In [ ]:
results = pd.DataFrame([
    result_A,
    result_B,
    result_C,
    result_D,
    result_E,
    result_F
])

results

,Run,Learning Rate,Epochs,LoRA r,Loss,Time (seconds)
0,run_A,0.0001,2,4,1.790226,223.457561
1,run_B,0.0002,2,4,1.742563,200.743055
2,run_C,0.0002,2,8,1.736244,199.452014
3,run_D_high_lr,0.0005,2,4,1.688054,198.637443
4,run_E_high_epochs,0.0002,3,4,1.700468,296.281636
5,run_F_low_rank,0.0002,2,2,1.744202,196.714201


## Final Overall Observations

- The experiment shows that changing hyperparameters affects both the training loss and the quality of generated responses.
- The same model, dataset, preprocessing and evaluation prompts were used for all runs, making the comparison fair.
- Increasing the learning rate from 1e-4 to 2e-4 improved the training loss while maintaining generally clear and relevant responses.
- The higher learning rate of 5e-4 gave the lowest training loss, but the responses showed more inconsistency, incorrect values and unrelated content.
- Increasing the LoRA rank from 4 to 8 gave only a small improvement in loss, while the response quality remained mostly similar.
- Lowering the LoRA rank to 2 reduced the number of trainable parameters, but some inconsistencies were still present in the generated responses.
- Increasing the number of epochs from 2 to 3 reduced the training loss, but increased the training time and may increase the risk of overfitting on a small dataset.
- Most runs generated relevant recipe-style responses, but extra information and unwanted continuation were common.
- Training loss alone is not enough to judge the model. Response quality, consistency, training time and overfitting should also be considered.
- Overall, LR = 2e-4, Epochs = 2 and LoRA r = 4 provided a reasonable balance between training loss, response quality and training time in this experiment.